<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/Tops__cloudflare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
# @title command install
!pip install polars scrapling patchright msgspec browserforge nest_asyncio beautifulsoup4 playwright xlsxwriter -q
!patchright install chromium
!patchright install-deps

In [ ]:
import asyncio
import nest_asyncio
import polars as pl
from datetime import date
from bs4 import BeautifulSoup
from patchright.async_api import async_playwright
import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
from playwright.async_api import async_playwright
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-04-16


In [ ]:
# ============================================================
# 1. Environment Setup
# ============================================================

# %%capture
# !pip install polars scrapling patchright msgspec browserforge nest_asyncio beautifulsoup4 -q
# !patchright install chromium
# !patchright install-deps

import asyncio
import nest_asyncio
import polars as pl
from datetime import date
from bs4 import BeautifulSoup
from patchright.async_api import async_playwright
from IPython.display import display

nest_asyncio.apply()


# ============================================================
# UDF: Data Prep & Transformation
# ============================================================

def process_tops_data(df: pl.DataFrame) -> pl.DataFrame:

    if df.is_empty():
        return df

    df = (
        df.with_columns([
            pl.col("promotion_price").cast(pl.Float64, strict=False),
            pl.col("original_price").cast(pl.Float64, strict=False)
        ])
        .with_columns(
            pl.when(
                pl.col("original_price").is_null() &
                pl.col("promotion_price").is_not_null()
            )
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            pl.when(
                pl.col("promotion_price") == pl.col("original_price")
            )
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )

    today = date.today().strftime("%Y-%m-%d")
    quant_pattern = r"(?i)(\d+)\s*(ML|G|KG|L)\b"

    return df.with_columns([
        pl.lit(today).alias("Date"),
        pl.col("name").str.split(" ").list.first().alias("Brand"),
        pl.col("name").str.extract(quant_pattern, 1)
            .cast(pl.Int64, strict=False).alias("Volume"),
        pl.col("name").str.extract(quant_pattern, 2)
            .str.to_uppercase().alias("Unit"),
        pl.lit("Tops").alias("Retailer")
    ])


# ============================================================
# Single URL Scraper
# ============================================================

async def scrape_tops_full_catalog(url: str) -> pl.DataFrame:

    extracted_data = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            )
        )

        await context.add_cookies([
            {'name': 'language', 'value': 'en', 'domain': '.tops.co.th', 'path': '/'},
            {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.tops.co.th', 'path': '/'}
        ])

        page = await context.new_page()

        print(f"Opening Tops: {url}")

        try:
            await page.goto(url, wait_until="load", timeout=90000)
            await asyncio.sleep(5)
        except Exception as e:
            print(f"Initial load error: {e}")
            await browser.close()
            return pl.DataFrame()

        previous_count = 0
        retries = 0

        for attempt in range(60):

            await page.keyboard.press("PageDown")
            await asyncio.sleep(2)

            await page.evaluate("window.scrollBy(0, 1000)")
            await asyncio.sleep(3)

            current_items = await page.query_selector_all(
                'div[class*="text-textblack"]'
            )
            current_count = len(current_items)

            if current_count > previous_count:
                print(f"Items detected: {current_count}...")
                previous_count = current_count
                retries = 0
            else:
                retries += 1
                await page.evaluate("window.scrollBy(0, -1500)")
                await asyncio.sleep(2)
                await page.keyboard.press("End")
                await asyncio.sleep(2)

            if retries >= 5:
                print("End of catalog or scroll limit reached.")
                break

        html = await page.content()
        soup = BeautifulSoup(html, "html.parser")

        name_tags = soup.find_all(
            "div",
            class_=lambda x: x and "text-textblack" in x
        )

        for nt in name_tags:

            name = nt.get_text(strip=True)

            parent = (
                nt.find_parent(
                    "div",
                    class_=lambda x: x and ("relative" in x or "item" in x)
                )
                or nt.parent
            )

            price_spans = parent.find_all(
                "span",
                class_="hidden lg:inline"
            )

            if len(price_spans) >= 2:
                promo = price_spans[0].get_text(strip=True).replace(',', '')
                orig = price_spans[1].get_text(strip=True).replace(',', '')
            elif len(price_spans) == 1:
                promo = None
                orig = price_spans[0].get_text(strip=True).replace(',', '')
            else:
                promo = orig = None

            extracted_data.append({
                "name": name,
                "promotion_price": promo,
                "original_price": orig
            })

        await browser.close()

    df_raw = pl.DataFrame(extracted_data)
    return process_tops_data(df_raw)


# ============================================================
# Multi URL Scraper ✅ FIXED
# ============================================================

async def scrape_tops_multi_url(urls: list[str]) -> pl.DataFrame:

    temp_df = pl.DataFrame()

    for url in urls:
        try:
            df = await scrape_tops_full_catalog(url)
            temp_df = pl.concat([temp_df, df])
        except Exception as e:
            print(f"Error with {url}: {e}")

    return temp_df

Tops has limited query MAX 40.

In [ ]:
# --- EXECUTION ---

urls_to_scrape = [
    "https://www.tops.co.th/en/household-and-pet/laundry/fabric-softener?brand=HYGIENE",
    "https://www.tops.co.th/en/household-and-pet/laundry/fabric-softener?brand=FINELINE",
    "https://www.tops.co.th/en/household-and-pet/laundry/liquid-detergent?brand=FINELINE",
    "https://www.tops.co.th/en/household-and-pet/laundry/liquid-detergent?brand=ATTACK%2CHYGIENE",
    "https://www.tops.co.th/en/household-and-pet/laundry/liquid-detergent?brand=PAO",
    "https://www.tops.co.th/en/household-and-pet/laundry/concentrated-fabric-softener?brand=HYGIENE",
    "https://www.tops.co.th/en/household-and-pet/laundry/concentrated-fabric-softener?brand=FINELINE",
    "https://www.tops.co.th/en/household-and-pet/laundry/powder-detergent?brand=ATTACK%2CPAO%2CPRO",
    "https://www.tops.co.th/en/household-and-pet/laundry/regular-fabric-softener?brand=HYGIENE%2CFINELINE",
    "https://www.tops.co.th/en/household-and-pet/laundry/gel-ball-detergent?brand=PAO",
    "https://www.tops.co.th/en/household-and-pet/dish-cleaner/dish-detergent?brand=LIPON+F"
]

# ✅ MUST await async function
tops_df = await scrape_tops_multi_url(urls_to_scrape)

# Display
if not tops_df.is_empty():
    display(tops_df)
    print(f"Grand Total Items Found: {len(tops_df)}")
else:
    print("No data collected.")

Opening Tops: https://www.tops.co.th/en/household-and-pet/laundry/fabric-softener?brand=HYGIENE
Items detected: 40...
End of catalog or scroll limit reached.
Opening Tops: https://www.tops.co.th/en/household-and-pet/laundry/fabric-softener?brand=FINELINE
Items detected: 39...
End of catalog or scroll limit reached.
Opening Tops: https://www.tops.co.th/en/household-and-pet/laundry/liquid-detergent?brand=FINELINE
Items detected: 34...
End of catalog or scroll limit reached.
Opening Tops: https://www.tops.co.th/en/household-and-pet/laundry/liquid-detergent?brand=ATTACK%2CHYGIENE
End of catalog or scroll limit reached.
Error with https://www.tops.co.th/en/household-and-pet/laundry/liquid-detergent?brand=ATTACK%2CHYGIENE: unable to append to a DataFrame of width 8 with a DataFrame of width 0
Opening Tops: https://www.tops.co.th/en/household-and-pet/laundry/liquid-detergent?brand=PAO
Items detected: 16...
End of catalog or scroll limit reached.
Opening Tops: https://www.tops.co.th/en/househo

name,promotion_price,original_price,Date,Brand,Volume,Unit,Retailer
str,f64,f64,str,str,i64,str,str
"""Hygiene Expert Care Concentrat…",109.0,129.0,"""2026-04-16""","""Hygiene""",1000,"""ML""","""Tops"""
"""Hygiene Expert Care Sunrise Ki…",109.0,129.0,"""2026-04-16""","""Hygiene""",1000,"""ML""","""Tops"""
"""Hygiene Fabric Softener Soft W…",109.0,129.0,"""2026-04-16""","""Hygiene""",500,"""ML""","""Tops"""
"""Hygiene Fabric Softener Violet…",109.0,129.0,"""2026-04-16""","""Hygiene""",1800,"""ML""","""Tops"""
"""Hygiene Expert Care Concentrat…",109.0,129.0,"""2026-04-16""","""Hygiene""",1000,"""ML""","""Tops"""
…,…,…,…,…,…,…,…
"""Lipon F Dish Wash Kaffir Lime …",145.0,165.0,"""2026-04-16""","""Lipon""",500,"""ML""","""Tops"""
"""Lipon F Dish Wash 500ml.""",145.0,165.0,"""2026-04-16""","""Lipon""",500,"""ML""","""Tops"""
"""Lipon F Dish Wash Japanese Yuz…",145.0,165.0,"""2026-04-16""","""Lipon""",475,"""ML""","""Tops"""


Grand Total Items Found: 252


In [ ]:
import polars as pl

def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. Swaps original and promotion if original < promotion.
    2. If original_price is missing, fills it with promotion_price.
    3. If promotion_price matches original_price, sets promotion to Null.
    """
    return (
        df.with_columns(
            # Create temporary columns to determine the true Max and Min
            # This automatically handles the "swap" and "missing original" logic
            pl.max_horizontal("original_price", "promotion_price").alias("original_price"),
            pl.min_horizontal("original_price", "promotion_price").alias("temp_promo")
        )
        .with_columns(
            # Finalize promotion_price: if it equals original_price, it's not a promotion
            pl.when(pl.col("temp_promo") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("temp_promo"))
            .alias("promotion_price")
        )
        .drop("temp_promo") # Clean up helper column
    )

# Usage:
df_prep_tops = re_evaluate_price(tops_df)

In [ ]:
# @title udf Transform
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Define the patterns (Centralized here so you only ever update them in one place!)
    quant_unit_pattern = r"(?i)(\d+)\s*(ML|G|KG|L)\b"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Extract Quantity (Pro-Tip: strict=False prevents the pipeline from crashing if a weird string can't be cast to an Integer)
        pl.col("name").str.extract(quant_unit_pattern, 1).cast(pl.Int16, strict=False).alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )



In [ ]:
df_trans_tops = parse_product_names(df_prep_tops, "Tops")

In [ ]:
df_trans_tops

name,promotion_price,original_price,Date,Brand,Volume,Unit,Retailer,Pack
str,f64,f64,str,str,i16,str,str,str
"""Hygiene Expert Care Concentrat…",109.0,129.0,"""2026-04-16""","""Hygiene""",1000,"""ML""","""Tops""",null
"""Hygiene Expert Care Sunrise Ki…",109.0,129.0,"""2026-04-16""","""Hygiene""",1000,"""ML""","""Tops""","""PACK 2"""
"""Hygiene Fabric Softener Soft W…",109.0,129.0,"""2026-04-16""","""Hygiene""",500,"""ML""","""Tops""","""PACK 3 FREE 1"""
"""Hygiene Fabric Softener Violet…",109.0,129.0,"""2026-04-16""","""Hygiene""",1800,"""ML""","""Tops""",null
"""Hygiene Expert Care Concentrat…",109.0,129.0,"""2026-04-16""","""Hygiene""",1000,"""ML""","""Tops""","""PACK"""
…,…,…,…,…,…,…,…,…
"""Lipon F Dish Wash Kaffir Lime …",145.0,165.0,"""2026-04-16""","""Lipon""",500,"""ML""","""Tops""",null
"""Lipon F Dish Wash 500ml.""",145.0,165.0,"""2026-04-16""","""Lipon""",500,"""ML""","""Tops""",null
"""Lipon F Dish Wash Japanese Yuz…",145.0,165.0,"""2026-04-16""","""Lipon""",475,"""ML""","""Tops""",null


In [ ]:
df_trans_tops.unique().write_excel(f"tops_{today_date}.xlsx")